In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential

train=pd.read_csv("engtamilTrain.csv", encoding='latin1', engine='python', sep=',', quotechar='"', on_bad_lines='skip')
train=train.drop(["Unnamed: 0"],axis=1)
english_sentences=train["en"]
tamil_sentence=train['ta']
english_sentences=english_sentences.head(1000)
tamil_sentences=tamil_sentence.head(1000)

In [7]:

def addSosEos(seriesSentence):
    sos_token = "<SOS>"
    eos_token = "<EOS>"

    statements_with_tokens = [f"{sos_token} {statement} {eos_token}" for statement in seriesSentence]

    english_sent=[]
    for statement in statements_with_tokens:
        english_sent.append(statement)
        print(statement)
    return english_sent


In [8]:
english_sent_SE=addSosEos(english_sentences)
tamil_sent_SE=addSosEos(tamil_sentences)

<SOS> MMA vice president Qazi Hussain Ahmad declared last month: 'We are not extremists.
 <EOS>
<SOS> Information has surfaced in recent years suggesting that Julius Rosenberg was involved in passing some form of intelligence to Soviet officials during the Second World War.
 <EOS>
<SOS> And Azor begat Sadoc; and Sadoc begat Achim; and Achim begat Eliud;
 <EOS>
<SOS> She says she knows what is going on, but can do nothing about it.
 <EOS>
<SOS> And be it indeed that I have erred, my error remains with myself.
 <EOS>
<SOS> Finally, the columnist fails to tell us who among the political leaders of the bourgeoisie, past and present, he counts among the paragons of morality.
 <EOS>
<SOS> These include the British Tamil Forum, La Maison du Tamil Eelam (France), the Canadian Tamil Congress, and the Swiss Tamil Forum.
 <EOS>
<SOS> Vijay accompanied with his wife and daughter enjoyed the film 'Anjathey'.
 <EOS>
<SOS> Both Musharraf and Vajpayee have exploited the current war drive to divert pub

In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [10]:
english_tokenizer = Tokenizer(filters="")
english_tokenizer.fit_on_texts(english_sent_SE)
english_vocab_size = len(english_tokenizer.word_index) + 1
english_sequences = english_tokenizer.texts_to_sequences(english_sent_SE)

In [11]:
tamil_tokenizer = Tokenizer(filters="")
tamil_tokenizer.fit_on_texts(tamil_sent_SE)
tamil_vocab_size = len(tamil_tokenizer.word_index) + 1
tamil_sequences = tamil_tokenizer.texts_to_sequences(tamil_sent_SE)

In [12]:
max_input_seq_length=20
max_output_seq_length=20

In [13]:
input_sequences = pad_sequences(english_sequences, maxlen=max_input_seq_length, padding='post')
output_sequences = pad_sequences(tamil_sequences, maxlen=max_output_seq_length, padding='post')

In [14]:
decoder_input_sequences = np.zeros_like(output_sequences)
decoder_input_sequences[:, 1:] = output_sequences[:, :-1]
decoder_input_sequences[:, 0] = tamil_tokenizer.word_index['<sos>']

decoder_output_sequences = np.eye(tamil_vocab_size)[output_sequences]


In [15]:
!pip install gensim
from gensim.models import Word2Vec

eng_model = Word2Vec(sentences=[s.split() for s in english_sent_SE], vector_size=100, window=5, min_count=1, workers=4)
eng_model.save('engmodel.bin')

tam_model = Word2Vec(sentences=[s.split() for s in tamil_sent_SE], vector_size=100, window=5, min_count=1, workers=4)
tam_model.save('tammodel.bin')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 69.2 MB/s eta 0:00:00


In [16]:
def create_embedding_matrix(word2vec_model, tokenizer, vocab_size):
    embedding_matrix = np.zeros((vocab_size, word2vec_model.vector_size))
    for word, i in tokenizer.word_index.items():
        try:
            embedding_vector = word2vec_model.wv[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            pass
    return embedding_matrix

eng_embedding_matrix = create_embedding_matrix(eng_model, english_tokenizer, english_vocab_size)
tam_embedding_matrix = create_embedding_matrix(tam_model, tamil_tokenizer, tamil_vocab_size)


In [17]:
eng_embedding_matrix.shape

(6979, 100)

In [18]:
tam_embedding_matrix.shape

(9922, 100)

In [19]:
def create_seq2seq_model(input_vocab_size, output_vocab_size, input_seq_length, output_seq_length, hidden_units, eng_embedding_matrix, tam_embedding_matrix):
    encoder_inputs = Input(shape=(input_seq_length,))
    encoder_embedding = Embedding(input_vocab_size, hidden_units, weights=[eng_embedding_matrix], trainable=False)(encoder_inputs)
    encoder_lstm, encoder_state_h, encoder_state_c = LSTM(hidden_units, return_state=True)(encoder_embedding)

    decoder_inputs = Input(shape=(output_seq_length,))
    decoder_embedding = Embedding(output_vocab_size, hidden_units, weights=[tam_embedding_matrix], trainable=False)(decoder_inputs)
    decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=[encoder_state_h, encoder_state_c])
    decoder_dense = Dense(output_vocab_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    return model


In [20]:
target_sequences = tf.keras.utils.to_categorical(output_sequences, num_classes=tamil_vocab_size)


In [21]:
model = create_seq2seq_model(english_vocab_size, tamil_vocab_size, max_input_seq_length, max_output_seq_length, 100, eng_embedding_matrix, tam_embedding_matrix)

In [22]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [23]:
batch_size = 32
epochs = 100
model.fit([input_sequences, output_sequences], decoder_output_sequences, batch_size=batch_size, epochs=epochs, validation_split=0.2)


Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 27s 863ms/step - accuracy: 0.2572 - loss: 8.9121 - val_accuracy: 0.2722 - val_loss: 8.3114
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 571ms/step - accuracy: 0.2686 - loss: 7.4204 - val_accuracy: 0.2663 - val_loss: 7.5303
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 21s 572ms/step - accuracy: 0.2684 - loss: 6.5786 - val_accuracy: 0.2722 - val_loss: 7.4854
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 16s 630ms/step - accuracy: 0.2692 - loss: 6.4216 - val_accuracy: 0.2722 - val_loss: 7.5333
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 537ms/step - accuracy: 0.2693 - loss: 6.3432 - val_accuracy: 0.2722 - val_loss: 7.5200
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 21s 552ms/step - accuracy: 0.2691 - loss: 6.2662 - val_accuracy: 0.2720 - val_loss: 7.4649
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 540ms/step - accuracy: 0.2688 - loss: 6.2113 - val_accuracy: 0.2722 - val_loss: 7.5687
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 13s 534ms/step - accuracy: 0.2691 - loss: 6.1681 - 

In [29]:
input_sentence = "<sos>Finally, the columnist fails to tell us who among the political leaders of the bourgeoisie, past and present, he counts among the paragons of morality<eos>"

input_sequence = english_tokenizer.texts_to_sequences([input_sentence])

input_sequence = pad_sequences(input_sequence, maxlen=max_input_seq_length, padding='post')

predictions = model.predict([input_sequence, np.zeros((1, max_output_seq_length))])

predicted_tokens = np.argmax(predictions, axis=-1)[0]

tamil_index_word = {i: w for w, i in tamil_tokenizer.word_index.items()}


decoded_sentence = []
for token in predicted_tokens:
    if token == 0:
        continue
    word = tamil_index_word.get(token)
    if word == '<eos>':
        break
    if word is not None:
        decoded_sentence.append(word)
    else:
        decoded_sentence.append('<unk>')

decoded_statement = ' '.join(decoded_sentence)

print(decoded_statement)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
à®à®µà®°à¯ à®à®¤à®©à¯ à®à®®à¯à®°à®¿à®à¯à® à®¨à¯ à®à®µà®©à¯à®à¯à®¯


In [25]:
predictions

array([[[2.61356559e-04, 7.63806794e-03, 6.75663992e-04, ...,
         7.59921264e-08, 6.30175379e-08, 7.10198123e-08],
        [6.88415821e-05, 2.46296531e-05, 4.81715705e-03, ...,
         1.27817415e-07, 1.02493026e-07, 1.13043463e-07],
        [1.14391063e-04, 7.85663087e-06, 1.83453935e-03, ...,
         1.34762203e-07, 1.16130849e-07, 1.29434767e-07],
        ...,
        [8.02547514e-01, 2.53654161e-05, 8.50708559e-02, ...,
         2.84511303e-09, 2.20457963e-09, 2.61184341e-09],
        [8.42543542e-01, 8.56993847e-06, 1.07850201e-01, ...,
         7.18589799e-10, 5.57976665e-10, 6.57989052e-10],
        [8.55857968e-01, 5.03498859e-06, 1.11637957e-01, ...,
         3.64268032e-10, 2.82421336e-10, 3.32720990e-10]]], dtype=float32)

In [26]:
predicted_tokens

array([  7,  22,   8,  87, 574,   2,   2,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0])

In [27]:
input_sequence

array([[   6,  569,   35,   41,  172,    1,   67,  452,    4,    1, 1118,
         226,    5, 1119,   16, 2057,  172,    1, 2058,    4]],
      dtype=int32)

In [28]:
input_sequence

array([[   6,  569,   35,   41,  172,    1,   67,  452,    4,    1, 1118,
         226,    5, 1119,   16, 2057,  172,    1, 2058,    4]],
      dtype=int32)